# Import


In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'


In [2]:
# Import
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import Dataset
from typing import Union, List, Tuple
from PIL import Image
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import os
import mlflow
import mlflow.pytorch
from datetime import datetime
from matplotlib.colors import ListedColormap
from skimage import io
import shutil


# MLflow run

In [3]:
from mlflow.exceptions import MlflowException
import datetime

# End MLflow run
mlflow.end_run()

# Function to generate a unique experiment name
def generate_experiment_name():
    return f"experiment-{datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"

# Function to create and set a new experiment
def set_experiment(experiment_name):
    try:
        experiment = mlflow.get_experiment_by_name(experiment_name)
        if experiment is None:
            mlflow.create_experiment(experiment_name)
            experiment = mlflow.get_experiment_by_name(experiment_name)
        mlflow.set_experiment(experiment_name)
    except MlflowException as e:
        print(f"Error setting experiment: {e}")
        
        
# Generate a unique experiment name and set the experiment
experiment_name = generate_experiment_name()
set_experiment(experiment_name)

# Verify the experiment is set correctly
print(f"Set experiment '{experiment_name}'")


# Start an MLflow run
mlflow.start_run(run_name=f"inference-{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Set experiment 'experiment-2024-07-23_11-33-58'


<ActiveRun: >

# Setup



In [4]:
dropout_rate = 0.5
mlflow.log_param("dropout_rate", dropout_rate)  # Adjust the value as needed

# Définition du modèle
class UNet_Light_RDN(nn.Module):
    def __init__(self, n_channels, n_classes, dropout_rate, bilinear=True):
        super(UNet_Light_RDN, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        self.inc = DoubleConv(n_channels, 32, dropout_rate)
        self.down1 = Down(32, 64, dropout_rate)
        self.down2 = Down(64, 128, dropout_rate)
        self.down3 = Down(128, 256, dropout_rate)
        self.down4 = Down(256, 256, dropout_rate)
        self.up1 = Up(512, 128, dropout_rate, bilinear)
        self.up2 = Up(256, 64, dropout_rate, bilinear)
        self.up3 = Up(128, 32, dropout_rate, bilinear)
        self.up4 = Up(64, 32, dropout_rate, bilinear)
        self.outc = OutConv(32, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, dropout_rate):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),  # Add dropout here
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate)  # Add another dropout here
        )

    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels, dropout_rate):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels, dropout_rate)
        )

    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, dropout_rate, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        else:
            self.up = nn.ConvTranspose2d(in_channels // 2, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels, dropout_rate)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = torch.tensor([x2.size()[2] - x1.size()[2]])
        diffX = torch.tensor([x2.size()[3] - x1.size()[3]])
        x1 = nn.functional.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)




# Load model


In [5]:

# Chargement du modèle
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = "C:/Users/n.vanderesse/Desktop/STAGE_Nolan/Git/IA-SeReOs/RDN.pth"
net = UNet_Light_RDN(n_channels=1, n_classes=3, dropout_rate=dropout_rate)
net.load_state_dict(torch.load(model_path, map_location=device))
# Load the model using MLflow
#model_uri = "runs:/<run_id>/model"  # Replace <run_id> with the actual run ID where the model was saved
#net = mlflow.pytorch.load_model(model_uri)
net.to(device)
net.eval()

# Vérifiez si le fichier du modèle existe
if not os.path.exists(model_path):
    print(f"Model file not found at {model_path}")
else:
    net = UNet_Light_RDN(n_channels=1, n_classes=3, dropout_rate=dropout_rate)
    net.load_state_dict(torch.load(model_path, map_location=device))
    net.to(device)
    net.eval()
    print("Model loaded successfully.")
    mlflow.pytorch.log_model(net, "model")

# Log parameters
mlflow.log_param("model_path", model_path)

def load_image(image_path):
    image = Image.open(image_path).convert('L')  # Convert to grayscale
    transform = transforms.ToTensor()  # Convert to tensor without resizing or normalization
    return transform(image).unsqueeze(0).to(device)  # Add batch dimension



Model loaded successfully.


# Inference

In [6]:
def run_inference(image_path):
    input_tensor = load_image(image_path)
    print(f"Input tensor shape: {input_tensor.shape}, dtype: {input_tensor.dtype}")  # Debugging line
    with torch.no_grad():
        output = net(input_tensor)
        print(f"Model output shape: {output.shape}, dtype: {output.dtype}")  # Debugging line
        pred = torch.argmax(output, dim=1).cpu().numpy()
    print(f"Prediction shape: {pred.shape}, unique values: {np.unique(pred)}")  # Debugging line
    return pred[0]  # Remove batch dimension

# Visualization

In [7]:
def visualize_slice(prediction):
    grayscale_map = {
        2: 255,  # Bone (white)
        1: 128,  # Dirt (gray)
        0: 0     # Air (black)
    }
    segmented_image = np.zeros_like(prediction, dtype=np.uint8)
    for class_value, grayscale_value in grayscale_map.items():
        segmented_image[prediction == class_value] = grayscale_value
    return segmented_image


def plot_results(img_3d, segmented_3d, slice_indices=None):
    print(f"Shape of img_3d: {img_3d.shape}")  # Debugging line
    print(f"Shape of segmented_3d: {segmented_3d.shape}")  # Debugging line

    middle_slice_index = img_3d.shape[0] // 2 if slice_indices is None else slice_indices[0]
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(img_3d[middle_slice_index], cmap='gray')
    ax[0].set_title('Input Image (Middle Slice)')
    ax[1].imshow(segmented_3d[middle_slice_index], cmap='gray')
    ax[1].set_title('Segmented Image (Middle Slice)')
    for a in ax:
        a.axis('off')
    plt.show()

    
    

# Directory inference (use for directories)

In [8]:


# Function to empty the destination directories
def empty_directory(directory):
    for filename in os.listdir(directory):
        file_path = os.path.join(directory, filename)
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)

# Function to copy files
def copy_files(file_paths, destination_dir):
    for file_path in file_paths:
        shutil.copy(file_path, destination_dir)

# Function to get the number of digits in the file names
def get_num_digits(directory):
    files = [f for f in os.listdir(directory) if f.endswith('.tif')]
    if not files:
        raise ValueError("No .tif files found in the directory")
    num_digits = len(files[0].split('_')[-1].split('.')[0])
    return num_digits

# Function to get image files by index range
def get_image_files_by_index(directory, start_index, end_index):
    num_digits = get_num_digits(directory)
    image_files = []
    for idx in range(start_index, end_index + 1):
        file_name = f"{directory}/deux_{idx:0{num_digits}}.tif"
        if os.path.exists(file_name):
            image_files.append(file_name)
        else:
            print(f"File not found: {file_name}")
    return image_files

# Directory inference
def get_image_files(directory, extensions=['.tif', '.png', '.jpg', '.jpeg']):
    image_files = [os.path.join(directory, f) for f in os.listdir(directory) if f.lower().endswith(tuple(extensions))]
    return image_files

def process_image_file(image_path, output_dir):
    prediction = run_inference(image_path)
    segmented_image = visualize_slice(prediction)
    output_path = os.path.join(output_dir, os.path.basename(image_path))
    Image.fromarray(segmented_image).save(output_path)
    return output_path

def process_image_directory(input_dir, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    image_files = get_image_files(input_dir)
    for image_path in image_files:
        print(f"Processing image: {image_path}")
        output_path = process_image_file(image_path, output_dir)
        mlflow.log_artifact(output_path)
        print(f"Segmented image saved to: {output_path}")

# Application

In [ ]:
# Application Block
def process_3d_image(image_path):
    # Load the 3D image
    img_3d = io.imread(image_path)
    print(f"Loaded 3D image with shape: {img_3d.shape}")  # Debugging line

    predictions = []
    for i in range(img_3d.shape[0]):
        print(f"Processing slice {i+1}/{img_3d.shape[0]}")
        slice_path = f"temp_slice_{i}.tif"
        Image.fromarray(img_3d[i]).save(slice_path)
        prediction = run_inference(slice_path)
        segmented_slice = visualize_slice(prediction)
        predictions.append(segmented_slice)
        os.remove(slice_path)

    segmented_3d = np.stack(predictions, axis=0)
    io.imsave('segmented_image_3d.tif', segmented_3d)
    mlflow.log_artifact('segmented_image_3d.tif')

    # Plot the middle slice for verification
    plot_results(img_3d, segmented_3d)

# Application
image_path = "D:/Donnees_pour_segmentation_RDN/data/2.tif"
mlflow.log_param("image_path", image_path)
process_3d_image(image_path)

# End the MLflow run
mlflow.end_run()


# Application for directory

In [9]:
# Get the list of image files
input_dir = "D:/Donnees_pour_segmentation_RDN/data/deux/Unseg"
inference_files = get_image_files_by_index(input_dir, start_index=800, end_index=850)


input_directory = "D:\Donnees_pour_segmentation_RDN\data\inference_test\inference_input"  
output_directory = "D:\Donnees_pour_segmentation_RDN\data\inference_test\inference_output" 

os.makedirs(input_directory, exist_ok=True)
os.makedirs(output_directory, exist_ok=True)

# Empty the destination directories
empty_directory(input_directory)
empty_directory(output_directory)

# Copy the files to the new directory
copy_files(inference_files, input_directory)
 
mlflow.log_param("input_directory", input_directory)
mlflow.log_param("output_directory", output_directory)
process_image_directory(input_directory, output_directory)

# End the MLflow run
mlflow.end_run()

Processing image: D:\Donnees_pour_segmentation_RDN\data\inference_test\inference_input\deux_0800.tif
Input tensor shape: torch.Size([1, 1, 414, 410]), dtype: torch.float32
Model output shape: torch.Size([1, 3, 414, 410]), dtype: torch.float32
Prediction shape: (1, 414, 410), unique values: [0 2]
Segmented image saved to: D:\Donnees_pour_segmentation_RDN\data\inference_test\inference_output\deux_0800.tif
Processing image: D:\Donnees_pour_segmentation_RDN\data\inference_test\inference_input\deux_0801.tif
Input tensor shape: torch.Size([1, 1, 414, 410]), dtype: torch.float32
Model output shape: torch.Size([1, 3, 414, 410]), dtype: torch.float32
Prediction shape: (1, 414, 410), unique values: [0 2]
Segmented image saved to: D:\Donnees_pour_segmentation_RDN\data\inference_test\inference_output\deux_0801.tif
Processing image: D:\Donnees_pour_segmentation_RDN\data\inference_test\inference_input\deux_0802.tif
Input tensor shape: torch.Size([1, 1, 414, 410]), dtype: torch.float32
Model output s